In [ ]:
# reorganize_and_explore.py
from pathlib import Path
import pandas as pd
import numpy as np
import keras
import matplotlib.pyplot as plt

# Use relative path from notebook location
# This works whether you're in notebooks/, scripts/, or project root
notebook_dir = Path.cwd()

# Find project root (where pyproject.toml is)
project_root = notebook_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:  # Reached filesystem root
        raise FileNotFoundError("Could not find project root (pyproject.toml not found)")

# Now use relative paths from project root
base_dir = project_root / "fall_detection_data"
processed_dir = base_dir / "processed"
models_dir = base_dir / "models"
models_dir.mkdir(exist_ok=True)
output_dir = models_dir

print(f"📂 Project root: {project_root}")
print(f"📂 Data directory: {base_dir}")
print(f"📂 Models directory: {models_dir}")
print()

print("=" * 80)
print("CURRENT DIRECTORY STRUCTURE")
print("=" * 80)

# Show current structure
for item in sorted(base_dir.iterdir()):
    if item.is_dir():
        print(f"\n📁 {item.name}/")
        # Show what's inside each directory
        sub_items = list(item.iterdir())[:5]
        for sub in sub_items:
            if sub.is_dir():
                file_count = len(list(sub.glob("*")))
                print(f"   📁 {sub.name}/ ({file_count} files)")
            else:
                print(f"   📄 {sub.name}")
        if len(list(item.iterdir())) > 5:
            print(f"   ... and {len(list(item.iterdir())) - 5} more")

print("\n" + "=" * 80)
print("PROPOSED REORGANIZATION")
print("=" * 80)

proposed_structure = """
fall_detection_data/
├── KFall/
│   ├── sensor_data/
│   │   ├── SA06/
│   │   │   ├── S06T01R01.csv  (KFall format: S##T##R##.csv)
│   │   │   ├── S06T02R01.csv
│   │   │   └── ...
│   │   └── SA07/ ...
│   └── labels/
│       ├── SA06_label.xlsx
│       └── SA07_label.xlsx ...
│
├── SisFall/
│   ├── SA01/
│   │   ├── D01_SA01_R01.txt  (SisFall format: <CODE>_<SUBJECT>_<TRIAL>.txt)
│   │   ├── F01_SA01_R01.txt
│   │   └── ...
│   ├── SA02/ ...
│   └── SE01/ ... (elderly subjects)
│
└── processed/
    ├── kfall_features.pkl
    ├── sisfall_features.pkl
    └── fused_dataset.pkl
"""

print(proposed_structure)

print("\n" + "=" * 80)
print("DATASET COMPARISON")
print("=" * 80)

# KFall structure
kfall_sensor = base_dir / "KFall" / "sensor_data"
if kfall_sensor.exists():
    kfall_subjects = sorted([d.name for d in kfall_sensor.iterdir() if d.is_dir()])
    sample_kfall = kfall_sensor / kfall_subjects[0]
    sample_kfall_file = list(sample_kfall.glob("*.csv"))[0]
    
    df_kfall = pd.read_csv(sample_kfall_file)
    
    print("\n📊 KFALL DATASET:")
    print(f"   Subjects: {len(kfall_subjects)} (SA06-SA38)")
    print(f"   Sampling Rate: 100 Hz (needs upsampling to 200 Hz)")
    print(f"   File Format: S##T##R##.csv")
    print(f"   Columns: {df_kfall.columns.tolist()}")
    print(f"   Data Shape (sample): {df_kfall.shape}")
    print(f"   Has Labels: ✅ Yes (temporal annotations in Excel files)")

# SisFall structure
sisfall_dir = base_dir / "SisFall"
if sisfall_dir.exists():
    sisfall_subjects = sorted([d.name for d in sisfall_dir.iterdir() if d.is_dir()])
    adults = [s for s in sisfall_subjects if s.startswith('SA')]
    elderly = [s for s in sisfall_subjects if s.startswith('SE')]
    
    sample_sisfall = sisfall_dir / adults[0]
    sample_sisfall_file = list(sample_sisfall.glob("*.txt"))[0]
    
    # Read SisFall file - more robust parsing
    try:
        # Method 1: Read line by line and parse manually
        with open(sample_sisfall_file, 'r') as f:
            lines = f.readlines()
        
        data = []
        for line in lines:
            # Remove semicolon and split by comma or whitespace
            line = line.strip().replace(';', '')
            values = line.replace(',', ' ').split()
            if len(values) == 9:  # Should have 9 columns
                data.append([float(v) for v in values])
        
        df_sisfall = pd.DataFrame(data)
        
        print("\n📊 SISFALL DATASET:")
        print(f"   Subjects: {len(sisfall_subjects)} total")
        print(f"     - Adults (SA): {len(adults)} (SA01-SA23)")
        print(f"     - Elderly (SE): {len(elderly)} (SE01-SE15)")
        print(f"   Sampling Rate: 200 Hz ✅")
        print(f"   File Format: <CODE>_<SUBJECT>_<TRIAL>.txt")
        print(f"   Columns: 9 (ADXL345: 0-2, ITG3200: 3-5, MMA8451Q: 6-8)")
        print(f"   Data Shape (sample): {df_sisfall.shape}")
        print(f"   Has Labels: ❌ No (must use Algorithm 1)")
        print(f"   Data Format: Raw bits (needs conversion to physical units)")
        
    except Exception as e:
        print(f"\n❌ Error reading SisFall file: {e}")
        print("   Will handle this in the preprocessing pipeline")

print("\n" + "=" * 80)
print("=" * 80)

print("\n📋 FROM KFALL (Table I):")
kfall_needed = {
    'T10': 'Stumble while walking',
    'T28': 'Vertical fall while walking (fainting)',
    'T30': 'Forward fall while walking (trip)',
    'T31': 'Forward fall while jogging (trip)',
    'T32': 'Forward fall while walking (slip)',
    'T33': 'Lateral fall while walking (slip)',
    'T34': 'Backward fall while walking (slip)'
}
for code, desc in kfall_needed.items():
    print(f"   {code}: {desc}")

print("\n📋 FROM SISFALL (Table I):")
print("\n   ADL Activities:")
sisfall_adl = {
    'D01': 'Walking slowly',
    'D02': 'Walking quickly',
    'D03': 'Jogging slowly',
    'D04': 'Jogging quickly',
    'D05': 'Walking upstairs/downstairs slowly',
    'D06': 'Walking upstairs/downstairs quickly',
    'D18': 'Stumble while walking'
}
for code, desc in sisfall_adl.items():
    print(f"   {code}: {desc}")

print("\n   Fall Activities:")
sisfall_falls = {
    'F01': 'Fall forward while walking (slip)',
    'F02': 'Fall backward while walking (slip)',
    'F03': 'Lateral fall while walking (slip)',
    'F04': 'Fall forward while walking (trip)',
    'F05': 'Fall forward while jogging (trip)',
    'F06': 'Vertical fall while walking (fainting)'
}
for code, desc in sisfall_falls.items():
    print(f"   {code}: {desc}")

print("\n" + "=" * 80)
print("NEXT STEPS")
print("=" * 80)
print("""
1. ✅ Data is properly organized
2. ⏭️  Implement preprocessing pipeline:
   - Load and convert SisFall raw bits to physical units
   - Upsample KFall from 100Hz to 200Hz
   - Apply Algorithm 1 for temporal segmentation
   - Extract features according to Table I
3. ⏭️  Z-score normalization and dataset fusion
4. ⏭️  Build and train FallNet""")


In [ ]:
# Load the newly processed data
X_data = np.load(processed_dir / "X_data.npy")
y_labels = np.load(processed_dir / "y_labels.npy")

# ============================================================================
# STEP 1: Merge Impact and Aftermath
# ============================================================================
print("Merging Impact and Aftermath classes...")
y_labels[y_labels == 7] = 6  # Change Aftermath (7) to Impact (6)

# ============================================================================
# STEP 2: Remove Fall_Recovery (NEW!)
# ============================================================================
print("\n" + "="*80)
print("REMOVING FALL_RECOVERY CLASS")
print("="*80)

from collections import Counter

# Show before
counts_before = Counter(y_labels)
print(f"\nBefore removal:")
print(f"  Total samples: {len(y_labels):,}")
print(f"  Fall_Recovery (class 4): {counts_before[4]} samples")

# Remove Fall_Recovery (class 4)
mask = y_labels != 4
X_data = X_data[mask]
y_labels_temp = y_labels[mask]

removed_count = (~mask).sum()
print(f"\n✅ Removed {removed_count} Fall_Recovery samples")

# Shift labels down (5→4, 6→5)
y_labels = y_labels_temp.copy()
y_labels[y_labels_temp > 4] -= 1  # Classes 5,6 become 4,5

print(f"\nAfter removal:")
print(f"  Total samples: {len(y_labels):,}")
print(f"  Removed: {removed_count} samples ({removed_count/(len(y_labels)+removed_count)*100:.2f}%)")

# ============================================================================
# STEP 3: Update label map (NOW 6 CLASSES: 0-5)
# ============================================================================
label_map = {
    'Walking': 0,
    'Jogging': 1,
    'Walking_stairs_updown': 2,
    'Stumble_while_walking': 3,
    'Fall_Initiation': 4,      # Was 5, now 4 ← SHIFTED DOWN!
    'Impact_Aftermath': 5,     # Was 6, now 5 ← SHIFTED DOWN!
}
reverse_label_map = {v: k for k, v in label_map.items()}

print(f"\n✅ Updated to 6 classes (0-5):")
for name, idx in sorted(label_map.items(), key=lambda x: x[1]):
    print(f"  Class {idx}: {name}")

y_categorical = keras.utils.to_categorical(y_labels, num_classes=6)  # ← HERE!
print(f"y_categorical shape: {y_categorical.shape}")

# ============================================================================
# DIAGNOSTICS
# ============================================================================
print("\n" + "="*80)
print("POST-REMOVAL DATA DIAGNOSTICS")
print("="*80)

# 1. Class distribution
class_counts = Counter(y_labels)
print("\n1. Class Distribution (6 classes):")
for cls_idx in sorted(class_counts.keys()):
    count = class_counts[cls_idx]
    pct = count / len(y_labels) * 100
    print(f"   Class {cls_idx} ({reverse_label_map[cls_idx]:30s}): {count:5d} ({pct:5.2f}%)")

# Calculate imbalance
max_count = max(class_counts.values())
min_count = min(class_counts.values())
print(f"\nImbalance ratio: {max_count/min_count:.2f}x (was 36.8x with Fall_Recovery)")

# 2. Per-class signal statistics
print("\n2. Per-Class Signal Statistics (Acc-Y axis):")
print(f"   {'Class':<35s} {'Mean':<10s} {'Std':<10s} {'Min':<10s} {'Max':<10s}")
print(f"   {'-'*75}")
for cls_idx in sorted(class_counts.keys()):
    class_samples = X_data[y_labels == cls_idx]
    acc_y = class_samples[:, :, 1]  # Y-axis acceleration
    
    mean_val = acc_y.mean()
    std_val = acc_y.std()
    min_val = acc_y.min()
    max_val = acc_y.max()
    
    print(f"   {reverse_label_map[cls_idx]:<35s} {mean_val:>8.4f}  {std_val:>8.4f}  {min_val:>8.2f}  {max_val:>8.2f}")

# 3. Variance ranking
print("\n3. Variance Ranking (Fall_Initiation should be #1):")
variances = []
for cls_idx in sorted(class_counts.keys()):
    class_samples = X_data[y_labels == cls_idx]
    acc_y_var = class_samples[:, :, 1].var()
    variances.append((reverse_label_map[cls_idx], acc_y_var, cls_idx))
variances.sort(key=lambda x: x[1], reverse=True)
for i, (name, var, idx) in enumerate(variances, 1):
    print(f"   {i}. {name:<35s}: {var:.4f}")

# 4. Visualize samples (update to 6 classes)
fig, axes = plt.subplots(3, 2, figsize=(15, 10))
axes = axes.flatten()
critical_classes = [
    label_map['Walking'],
    label_map['Fall_Initiation'],
    label_map['Impact_Aftermath'],
    label_map['Stumble_while_walking'],
    label_map['Jogging'],
    label_map['Walking_stairs_updown']
]
for i, cls_idx in enumerate(critical_classes):
    if cls_idx in class_counts:
        sample_idx = np.where(y_labels == cls_idx)[0][0]
        sample_data = X_data[sample_idx]
        
        time = np.arange(200) / 200
        axes[i].plot(time, sample_data[:, 0], label='Acc-X', alpha=0.7, linewidth=1)
        axes[i].plot(time, sample_data[:, 1], label='Acc-Y', alpha=0.7, linewidth=1)
        axes[i].plot(time, sample_data[:, 2], label='Acc-Z', alpha=0.7, linewidth=1)
        
        axes[i].set_title(f'{reverse_label_map[cls_idx]}', fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Time (s)')
        axes[i].set_ylabel('Normalized Acc')
        axes[i].legend(fontsize=8)
        axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("✅ DATA READY FOR TRAINING (6 CLASSES)")
print("="*80)

In [ ]:
# %% [markdown]
## Debug: Test SNN Evaluation

# %%
print("\n" + "="*80)
print("DEBUG: Testing SNN Model Loading & Evaluation")
print("="*80)

# Load first fold's model
test_path = snn_dir / "snn_fold_1.pth"

if test_path.exists():
    print(f"\nLoading test model: {test_path}")
    
    # Load model
    test_model = MicroCNN_SNN(num_classes=6, num_steps=25).to(device)
    test_model.load_state_dict(torch.load(test_path, map_location=device))
    test_model.eval()
    
    # Get first fold validation data
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    _, val_idx = next(skf.split(X_data, y_labels))
    
    X_test = X_data[val_idx][:100]  # Just 100 samples
    y_test = y_labels[val_idx][:100]
    
    # Convert to PyTorch
    X_test_torch = torch.FloatTensor(X_test).permute(0, 2, 1).to(device)
    y_test_torch = torch.LongTensor(y_test).to(device)
    
    print(f"\nTest data shape: {X_test_torch.shape}")
    
    # Forward pass
    with torch.no_grad():
        spk_out, mem_out = test_model(X_test_torch)
    
    print(f"\nOutput shapes:")
    print(f"  spk_out: {spk_out.shape}")
    print(f"  mem_out: {mem_out.shape}")
    
    # Method 1: Sum spikes
    spk_sum = spk_out.sum(dim=0)
    _, pred_method1 = spk_sum.max(1)
    
    print(f"\nMethod 1 (sum spikes):")
    print(f"  spk_sum shape: {spk_sum.shape}")
    print(f"  predictions shape: {pred_method1.shape}")
    print(f"  Accuracy: {(pred_method1 == y_test_torch).float().mean():.4f}")
    
    # Method 2: Average membrane potential
    mem_avg = mem_out.mean(dim=0)
    _, pred_method2 = mem_avg.max(1)
    
    print(f"\nMethod 2 (avg membrane):")
    print(f"  mem_avg shape: {mem_avg.shape}")
    print(f"  predictions shape: {pred_method2.shape}")
    print(f"  Accuracy: {(pred_method2 == y_test_torch).float().mean():.4f}")
    
    # Check which method was used in training
    print(f"\n{'='*80}")
    print("EXPECTED: Method 1 accuracy should be ~88%")
    print("If Method 1 is ~35%, there's a bug in the model itself!")
    print("="*80)
else:
    print(f"\n❌ Test model not found: {test_path}")

# %%

In [ ]:
# Make sure these match what you used for training!
X_data = np.load(processed_dir / "X_data_6class.npy")
y_labels = np.load(processed_dir / "y_labels_6class.npy")

print(f"Data shapes: X={X_data.shape}, y={y_labels.shape}")
print(f"Unique labels: {np.unique(y_labels)}")
print(f"Expected: [0, 1, 2, 3, 4, 5] (6 classes)")

In [ ]:
# Load data
X_data = np.load(processed_dir / "X_data_6class.npy")
y_labels = np.load(processed_dir / "y_labels_6class.npy")

# ✅ ADD THIS DEBUG:
print(f"\n🔍 DEBUG - Data Format Check:")
print(f"   Loaded X_data shape: {X_data.shape}")
print(f"   Expected: (16732, 200, 6)")

# Check if it's already transposed
if X_data.shape == (16732, 6, 200):
    print("   ⚠️  Data is ALREADY transposed!")
    print("   ⚠️  Don't permute again or you'll transpose it TWICE!")
elif X_data.shape == (16732, 200, 6):
    print("   ✅ Data format is correct")
else:
    print(f"   ❌ Unexpected shape!")

In [ ]:
# Right after: model_snn.load_state_dict(torch.load(snn_path, map_location=device))

print(f"\n🔬 DEFINITIVE TEST - Fold {fold}")
print(f"="*80)

# Get the EXACT same validation data as training
X_val_torch = torch.FloatTensor(X_val).permute(0, 2, 1).to(device)

# Test on first 100 samples
test_X = X_val_torch[:100]
test_y = torch.LongTensor(y_val[:100]).to(device)

with torch.no_grad():
    spk_out, mem_out = model_snn(test_X)
    spk_sum = spk_out.sum(dim=0)
    _, predicted = spk_sum.max(1)
    
    test_acc = (predicted == test_y).float().mean()
    
    print(f"   Test on first 100 samples: {test_acc:.4f}")
    print(f"   Expected: ~0.88 (should match training)")
    print(f"   If this is ~0.35, the MODEL is broken")
    print(f"   If this is ~0.88, the EVALUATION LOOP is broken")

print(f"="*80)

In [ ]:
print("\n" + "="*80)
print("FULL DEBUG CHECK")
print("="*80)

# 1. Data files
print("\n1. Data Files:")
print(f"   X_data file: {processed_dir / 'X_data_6class.npy'}")
print(f"   File exists: {(processed_dir / 'X_data_6class.npy').exists()}")
print(f"   Loaded shape: {X_data.shape}")
print(f"   Unique labels: {np.unique(y_labels)}")
print(f"   Label counts: {np.bincount(y_labels)}")

# 2. K-fold
print("\n2. K-Fold Configuration:")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
first_fold_train, first_fold_val = next(skf.split(X_data, y_labels))
print(f"   Random state: 42")
print(f"   First fold val size: {len(first_fold_val)}")
print(f"   First fold val indices (first 10): {first_fold_val[:10]}")

# 3. Model file
print("\n3. Model File:")
snn_path = snn_dir / "snn_fold_1.pth"
print(f"   SNN path: {snn_path}")
print(f"   Exists: {snn_path.exists()}")
print(f"   Size: {snn_path.stat().st_size / 1024:.2f} KB")

# 4. Quick accuracy test
print("\n4. Quick Accuracy Test (Fold 1, first 100 samples):")
model_test = MicroCNN_SNN(6, 25).to(device)
model_test.load_state_dict(torch.load(snn_path, map_location=device))
model_test.eval()

X_test = torch.FloatTensor(X_data[first_fold_val[:100]]).permute(0, 2, 1).to(device)
y_test = torch.LongTensor(y_labels[first_fold_val[:100]]).to(device)

with torch.no_grad():
    spk, mem = model_test(X_test)
    _, pred = spk.sum(0).max(1)
    acc = (pred == y_test).float().mean()

print(f"   Accuracy: {acc:.4f}")
print(f"   Expected: ~0.88")

if acc < 0.5:
    print("\n❌ ACCURACY IS TERRIBLE! Possible causes:")
    print("   - Loading wrong data file")
    print("   - Loading wrong model file")  
    print("   - Data preprocessing mismatch")
else:
    print("\n✅ Accuracy looks good! Issue is in the evaluation loop.")

print("="*80)

In [ ]:
# %% [markdown]
# # Quantize Micro-CNN Models and Verify Accuracy

# %%
import numpy as np
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("MICRO-CNN QUANTIZATION PIPELINE")
print("="*80)

# Setup paths
current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:
        project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
        break

data_dir = project_root / "fall_detection_data"
processed_dir = data_dir / "processed"
models_dir = data_dir / "models"
micro_cnn_dir = models_dir / "micro_cnn"
quantized_dir = models_dir / "quantized"

# Create quantized directory
quantized_dir.mkdir(exist_ok=True)

print(f"\n📂 Directories:")
print(f"   Source models: {micro_cnn_dir}")
print(f"   Quantized output: {quantized_dir}")

# Load data
X_data = np.load(processed_dir / "X_data_6class.npy")
y_labels = np.load(processed_dir / "y_labels_6class.npy")

with open(processed_dir / "label_map_6class.json", 'r') as f:
    label_map = json.load(f)

reverse_label_map = {v: k for k, v in label_map.items()}

print(f"\n📊 Data loaded:")
print(f"   X shape: {X_data.shape}")
print(f"   y shape: {y_labels.shape}")
print(f"   Classes: {list(label_map.keys())}")

# %% [markdown]
## Representative Dataset Generator

# %%
def representative_dataset_generator(X_data, num_samples=1000):
    """
    Generate representative dataset for quantization calibration
    
    Args:
        X_data: Full dataset [N, time, features]
        num_samples: Number of samples to use for calibration
    
    Yields:
        Single samples for calibration
    """
    # Randomly sample from dataset
    indices = np.random.choice(len(X_data), size=min(num_samples, len(X_data)), replace=False)
    
    for idx in indices:
        # Yield as batch of 1
        sample = X_data[idx:idx+1].astype(np.float32)
        yield [sample]

print("✅ Representative dataset generator defined")

# %% [markdown]
## Quantization Functions

# %%
def quantize_model_int8(model, X_calibration, output_path):
    """
    Quantize model to INT8
    
    Args:
        model: Keras model
        X_calibration: Calibration data
        output_path: Where to save .tflite file
    
    Returns:
        Path to saved model
    """
    print(f"\n  Converting to INT8...")
    
    # Convert to TFLite
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    # Enable INT8 quantization
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    
    # Provide representative dataset
    converter.representative_dataset = lambda: representative_dataset_generator(X_calibration, num_samples=1000)
    
    # Ensure INT8 quantization
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.float32  # Input remains float32
    converter.inference_output_type = tf.float32  # Output remains float32
    
    # Convert
    tflite_model = converter.convert()
    
    # Save
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    
    file_size = output_path.stat().st_size / 1024
    print(f"  ✅ INT8 model saved: {file_size:.2f} KB")
    
    return output_path


def quantize_model_float16(model, output_path):
    """
    Quantize model to Float16
    
    Args:
        model: Keras model
        output_path: Where to save .tflite file
    
    Returns:
        Path to saved model
    """
    print(f"\n  Converting to Float16...")
    
    # Convert to TFLite
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    # Enable Float16 quantization
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    
    # Convert
    tflite_model = converter.convert()
    
    # Save
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    
    file_size = output_path.stat().st_size / 1024
    print(f"  ✅ Float16 model saved: {file_size:.2f} KB")
    
    return output_path


def evaluate_tflite_model(tflite_path, X_test, y_test):
    """
    Evaluate TFLite model accuracy
    
    Args:
        tflite_path: Path to .tflite file
        X_test: Test data
        y_test: Test labels
    
    Returns:
        Dictionary with metrics
    """
    # Load TFLite model
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    
    # Get input/output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Run inference
    predictions = []
    
    for i in range(len(X_test)):
        # Prepare input
        input_data = X_test[i:i+1].astype(np.float32)
        
        # Set input
        interpreter.set_tensor(input_details[0]['index'], input_data)
        
        # Run inference
        interpreter.invoke()
        
        # Get output
        output_data = interpreter.get_tensor(output_details[0]['index'])
        pred = np.argmax(output_data[0])
        predictions.append(pred)
    
    predictions = np.array(predictions)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions, average='weighted', zero_division=0)
    recall = recall_score(y_test, predictions, average='weighted', zero_division=0)
    f1 = f1_score(y_test, predictions, average='weighted', zero_division=0)
    
    # Fall_Initiation recall
    fall_init_idx = label_map["Fall_Initiation"]
    fall_init_mask = y_test == fall_init_idx
    if fall_init_mask.sum() > 0:
        fall_init_recall = (predictions[fall_init_mask] == fall_init_idx).mean()
    else:
        fall_init_recall = 0.0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'fall_init_recall': fall_init_recall,
        'predictions': predictions
    }

print("✅ Quantization functions defined")

# %% [markdown]
## Quantize All Fold Models

# %%
# K-Fold split (same as training)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
results = {
    'fp32': [],
    'float16': [],
    'int8': []
}

print("\n" + "="*80)
print("QUANTIZING ALL FOLD MODELS")
print("="*80)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/5")
    print(f"{'='*80}")
    
    # Load original model
    model_path = micro_cnn_dir / f"micro_cnn_fold_{fold}.keras"
    
    if not model_path.exists():
        print(f"⚠️  Model not found: {model_path}")
        continue
    
    print(f"Loading: {model_path.name}")
    model = keras.models.load_model(model_path)
    
    # Get validation data
    X_val = X_data[val_idx]
    y_val = y_labels[val_idx]
    
    # Get training data for calibration
    X_train = X_data[train_idx]
    
    print(f"Validation samples: {len(X_val):,}")
    print(f"Calibration samples: {len(X_train):,}")
    
    # ================================================================
    # 1. Evaluate FP32 (baseline)
    # ================================================================
    print(f"\n--- FP32 Baseline ---")
    y_pred_fp32 = np.argmax(model.predict(X_val, verbose=0), axis=1)
    
    fp32_metrics = {
        'fold': fold,
        'accuracy': accuracy_score(y_val, y_pred_fp32),
        'fall_init_recall': (y_pred_fp32[y_val == label_map["Fall_Initiation"]] == label_map["Fall_Initiation"]).mean() if (y_val == label_map["Fall_Initiation"]).sum() > 0 else 0.0
    }
    
    results['fp32'].append(fp32_metrics)
    print(f"  Accuracy: {fp32_metrics['accuracy']:.4f} ({fp32_metrics['accuracy']*100:.2f}%)")
    
    # ================================================================
    # 2. Quantize to Float16
    # ================================================================
    print(f"\n--- Float16 Quantization ---")
    float16_path = quantized_dir / f"micro_cnn_fold_{fold}_float16.tflite"
    quantize_model_float16(model, float16_path)
    
    # Evaluate
    print(f"  Evaluating Float16...")
    float16_metrics = evaluate_tflite_model(float16_path, X_val, y_val)
    float16_metrics['fold'] = fold
    results['float16'].append(float16_metrics)
    
    print(f"  Accuracy: {float16_metrics['accuracy']:.4f} ({float16_metrics['accuracy']*100:.2f}%)")
    print(f"  Drop from FP32: {(fp32_metrics['accuracy'] - float16_metrics['accuracy'])*100:.2f}%")
    
    # ================================================================
    # 3. Quantize to INT8
    # ================================================================
    print(f"\n--- INT8 Quantization ---")
    int8_path = quantized_dir / f"micro_cnn_fold_{fold}_int8.tflite"
    quantize_model_int8(model, X_train, int8_path)
    
    # Evaluate
    print(f"  Evaluating INT8...")
    int8_metrics = evaluate_tflite_model(int8_path, X_val, y_val)
    int8_metrics['fold'] = fold
    results['int8'].append(int8_metrics)
    
    print(f"  Accuracy: {int8_metrics['accuracy']:.4f} ({int8_metrics['accuracy']*100:.2f}%)")
    print(f"  Drop from FP32: {(fp32_metrics['accuracy'] - int8_metrics['accuracy'])*100:.2f}%")
    
    # Summary for this fold
    print(f"\n  Fold {fold} Summary:")
    print(f"    FP32:    {fp32_metrics['accuracy']*100:.2f}%")
    print(f"    Float16: {float16_metrics['accuracy']*100:.2f}% (↓{(fp32_metrics['accuracy'] - float16_metrics['accuracy'])*100:.2f}%)")
    print(f"    INT8:    {int8_metrics['accuracy']*100:.2f}% (↓{(fp32_metrics['accuracy'] - int8_metrics['accuracy'])*100:.2f}%)")

# %% [markdown]
## Results Summary

# %%
print("\n" + "="*80)
print("QUANTIZATION RESULTS SUMMARY")
print("="*80)

# Calculate statistics
fp32_accs = [r['accuracy'] for r in results['fp32']]
float16_accs = [r['accuracy'] for r in results['float16']]
int8_accs = [r['accuracy'] for r in results['int8']]

fp32_fall = [r['fall_init_recall'] for r in results['fp32']]
float16_fall = [r['fall_init_recall'] for r in results['float16']]
int8_fall = [r['fall_init_recall'] for r in results['int8']]

# Create comparison table
comparison_df = pd.DataFrame({
    'Model': ['FP32', 'Float16', 'INT8'],
    'Accuracy': [
        f"{np.mean(fp32_accs):.4f} ± {np.std(fp32_accs):.4f}",
        f"{np.mean(float16_accs):.4f} ± {np.std(float16_accs):.4f}",
        f"{np.mean(int8_accs):.4f} ± {np.std(int8_accs):.4f}"
    ],
    'Accuracy %': [
        f"{np.mean(fp32_accs)*100:.2f}%",
        f"{np.mean(float16_accs)*100:.2f}%",
        f"{np.mean(int8_accs)*100:.2f}%"
    ],
    'Fall_Init Recall': [
        f"{np.mean(fp32_fall)*100:.2f}%",
        f"{np.mean(float16_fall)*100:.2f}%",
        f"{np.mean(int8_fall)*100:.2f}%"
    ],
    'Size (est.)': ['~553 KB', '~85 KB', '~56 KB'],
    'Arduino': ['❌', '✅', '✅']
})

print("\n")
print(comparison_df.to_string(index=False))

print(f"\n{'='*80}")
print("ACCURACY DROPS FROM FP32")
print(f"{'='*80}")
print(f"Float16: {(np.mean(fp32_accs) - np.mean(float16_accs))*100:.2f}%")
print(f"INT8:    {(np.mean(fp32_accs) - np.mean(int8_accs))*100:.2f}%")

# %% [markdown]
## Detailed Per-Fold Comparison

# %%
print("\n" + "="*80)
print("PER-FOLD ACCURACY COMPARISON")
print("="*80)

per_fold_df = pd.DataFrame({
    'Fold': [r['fold'] for r in results['fp32']],
    'FP32': [f"{r['accuracy']*100:.2f}%" for r in results['fp32']],
    'Float16': [f"{r['accuracy']*100:.2f}%" for r in results['float16']],
    'INT8': [f"{r['accuracy']*100:.2f}%" for r in results['int8']],
    'Float16_Drop': [f"{(results['fp32'][i]['accuracy'] - results['float16'][i]['accuracy'])*100:.2f}%" for i in range(5)],
    'INT8_Drop': [f"{(results['fp32'][i]['accuracy'] - results['int8'][i]['accuracy'])*100:.2f}%" for i in range(5)]
})

print("\n")
print(per_fold_df.to_string(index=False))

# %% [markdown]
## Visualization

# %%
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Accuracy by quantization type
model_types = ['FP32', 'Float16', 'INT8']
mean_accs = [np.mean(fp32_accs)*100, np.mean(float16_accs)*100, np.mean(int8_accs)*100]
std_accs = [np.std(fp32_accs)*100, np.std(float16_accs)*100, np.std(int8_accs)*100]

colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = axes[0].bar(model_types, mean_accs, yerr=std_accs, capsize=5, 
                   color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

axes[0].set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
axes[0].set_title('Quantization Impact on Accuracy', fontsize=13, fontweight='bold')
axes[0].set_ylim([85, 100])
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, acc in zip(bars, mean_accs):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{acc:.2f}%',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Plot 2: Per-fold comparison
x = np.arange(5)
width = 0.25

axes[1].bar(x - width, [r['accuracy']*100 for r in results['fp32']], width, 
           label='FP32', color='#3498db', alpha=0.7)
axes[1].bar(x, [r['accuracy']*100 for r in results['float16']], width, 
           label='Float16', color='#2ecc71', alpha=0.7)
axes[1].bar(x + width, [r['accuracy']*100 for r in results['int8']], width, 
           label='INT8', color='#e74c3c', alpha=0.7)

axes[1].set_xlabel('Fold', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Accuracy by Fold and Quantization', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{i+1}' for i in range(5)])
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([85, 100])

plt.tight_layout()
plt.savefig(quantized_dir / 'quantization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Plot saved: {quantized_dir / 'quantization_comparison.png'}")

# %% [markdown]
## Save Summary Report

# %%
summary = f"""
MICRO-CNN QUANTIZATION SUMMARY
{'='*80}

Models Quantized: 5 folds
Output Directory: {quantized_dir}

RESULTS (5-Fold Average):

FP32 (Baseline):
  Accuracy:        {np.mean(fp32_accs):.4f} ± {np.std(fp32_accs):.4f} ({np.mean(fp32_accs)*100:.2f}%)
  Fall_Init Recall: {np.mean(fp32_fall):.4f} ± {np.std(fp32_fall):.4f} ({np.mean(fp32_fall)*100:.2f}%)
  Size:            ~553 KB
  Arduino:         ❌ Too large

Float16 (Quantized):
  Accuracy:        {np.mean(float16_accs):.4f} ± {np.std(float16_accs):.4f} ({np.mean(float16_accs)*100:.2f}%)
  Fall_Init Recall: {np.mean(float16_fall):.4f} ± {np.std(float16_fall):.4f} ({np.mean(float16_fall)*100:.2f}%)
  Size:            ~85 KB
  Arduino:         ✅ Fits
  Drop from FP32:  {(np.mean(fp32_accs) - np.mean(float16_accs))*100:.2f}%

INT8 (Quantized):
  Accuracy:        {np.mean(int8_accs):.4f} ± {np.std(int8_accs):.4f} ({np.mean(int8_accs)*100:.2f}%)
  Fall_Init Recall: {np.mean(int8_fall):.4f} ± {np.std(int8_fall):.4f} ({np.mean(int8_fall)*100:.2f}%)
  Size:            ~56 KB
  Arduino:         ✅ Fits
  Drop from FP32:  {(np.mean(fp32_accs) - np.mean(int8_accs))*100:.2f}%

DEPLOYMENT RECOMMENDATION:
  - For best accuracy: Float16 ({np.mean(float16_accs)*100:.2f}%, 85 KB)
  - For smallest size: INT8 ({np.mean(int8_accs)*100:.2f}%, 56 KB)
  - Both fit on Arduino Nano 33 BLE Sense

FILES CREATED:
"""

# Add file list
for fold in range(1, 6):
    summary += f"\n  Fold {fold}:"
    summary += f"\n    - micro_cnn_fold_{fold}_float16.tflite"
    summary += f"\n    - micro_cnn_fold_{fold}_int8.tflite"

with open(quantized_dir / 'quantization_summary.txt', 'w') as f:
    f.write(summary)

print(summary)
print(f"\n✅ Summary saved: {quantized_dir / 'quantization_summary.txt'}")

# Save detailed results to JSON
detailed_results = {
    'fp32': results['fp32'],
    'float16': results['float16'],
    'int8': results['int8'],
    'summary': {
        'fp32_mean_acc': float(np.mean(fp32_accs)),
        'float16_mean_acc': float(np.mean(float16_accs)),
        'int8_mean_acc': float(np.mean(int8_accs)),
        'float16_drop': float((np.mean(fp32_accs) - np.mean(float16_accs))*100),
        'int8_drop': float((np.mean(fp32_accs) - np.mean(int8_accs))*100)
    }
}

# Convert numpy types to Python types for JSON serialization
for model_type in ['fp32', 'float16', 'int8']:
    for result in detailed_results[model_type]:
        for key, value in result.items():
            if isinstance(value, np.ndarray):
                result[key] = value.tolist()
            elif isinstance(value, (np.float32, np.float64)):
                result[key] = float(value)

with open(quantized_dir / 'quantization_results.json', 'w') as f:
    json.dump(detailed_results, f, indent=2)

print(f"✅ Detailed results saved: {quantized_dir / 'quantization_results.json'}")

print("\n" + "="*80)
print("QUANTIZATION COMPLETE!")
print("="*80)
print(f"\n✅ All models quantized and saved to: {quantized_dir}")
print(f"✅ {len(results['float16'])} Float16 models created")
print(f"✅ {len(results['int8'])} INT8 models created")
print(f"✅ Ready for Arduino deployment!")


In [ ]:
# %% [markdown]
# # Compare Micro-CNN vs SNN Accuracy
# Load all fold models and compare performance

# %%
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate

print("="*80)
print("MICRO-CNN vs SNN ACCURACY COMPARISON")
print("="*80)

# Setup paths
current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:
        project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
        break

data_dir = project_root / "fall_detection_data"
processed_dir = data_dir / "processed"
models_dir = data_dir / "models"
micro_cnn_dir = models_dir / "micro_cnn"
snn_dir = models_dir / "snn"

print(f"\n📂 Directories:")
print(f"   Micro-CNN: {micro_cnn_dir}")
print(f"   SNN:       {snn_dir}")

# Check what files exist
if micro_cnn_dir.exists():
    cnn_files = list(micro_cnn_dir.glob("*.keras"))
    print(f"\n   Found {len(cnn_files)} CNN models:")
    for f in sorted(cnn_files):
        print(f"     - {f.name}")
else:
    print(f"   ⚠️  {micro_cnn_dir} doesn't exist")

if snn_dir.exists():
    snn_files = list(snn_dir.glob("*.pth"))
    print(f"\n   Found {len(snn_files)} SNN models:")
    for f in sorted(snn_files):
        print(f"     - {f.name}")
else:
    print(f"   ⚠️  {snn_dir} doesn't exist")

# Load data
X_data = np.load(processed_dir / "X_data_6class.npy")
y_labels = np.load(processed_dir / "y_labels_6class.npy")

with open(processed_dir / "label_map_6class.json", 'r') as f:
    label_map = json.load(f)

reverse_label_map = {v: k for k, v in label_map.items()}

print(f"\n📊 Data:")
print(f"   X shape: {X_data.shape}")
print(f"   y shape: {y_labels.shape}")

# %% [markdown]
## SNN Architecture (for loading)

# %%
class MicroCNN_SNN(nn.Module):
    """Clean SNN implementation without manual detaching"""
    
    def __init__(self, num_classes=6, num_steps=25, beta=0.95, threshold=1.0):
        super().__init__()
        
        self.num_classes = num_classes
        self.num_steps = num_steps
        
        # Surrogate gradient for backprop
        spike_grad = surrogate.fast_sigmoid(slope=25)
        
        # Block 1: Conv -> BN -> LIF -> Pool
        self.conv1 = nn.Conv1d(6, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool1 = nn.MaxPool1d(2)
        
        # Block 2: Conv -> BN -> LIF -> Pool
        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool2 = nn.MaxPool1d(2)
        
        # Block 3: Conv -> BN -> LIF -> Pool
        self.conv3 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        self.pool3 = nn.MaxPool1d(2)
        
        # Global pooling
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(128, 64)
        self.lif4 = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
        
        self.fc2 = nn.Linear(64, num_classes)
        self.lif_out = snn.Leaky(beta=beta, spike_grad=spike_grad, threshold=threshold)
    
    def forward(self, x):
        """
        Args:
            x: [batch, channels, time_steps] input tensor
        
        Returns:
            spk_rec: [num_steps, batch, num_classes] spike recordings
            mem_rec: [num_steps, batch, num_classes] membrane recordings
        """
        batch_size = x.size(0)
        
        # Initialize membrane potentials
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()
        mem_out = self.lif_out.init_leaky()
        
        # Recording lists
        spk_rec = []
        mem_rec = []
        # Process over time steps
        for step in range(self.num_steps):
            # Block 1
            cur1 = self.bn1(self.conv1(x))
            spk1, mem1 = self.lif1(cur1, mem1)
            spk1 = self.pool1(spk1)
            
            # Block 2
            cur2 = self.bn2(self.conv2(spk1))
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2 = self.pool2(spk2)
            
            # Block 3
            cur3 = self.bn3(self.conv3(spk2))
            spk3, mem3 = self.lif3(cur3, mem3)
            spk3 = self.pool3(spk3)
            
            # Global average pooling
            spk3 = self.global_pool(spk3).squeeze(-1)
            
            # FC1
            cur4 = self.fc1(spk3)
            spk4, mem4 = self.lif4(cur4, mem4)
            
            # Output layer
            cur_out = self.fc2(spk4)
            spk_out, mem_out = self.lif_out(cur_out, mem_out)
            
            # Record
            spk_rec.append(spk_out)
            mem_rec.append(mem_out)
        
        # Stack: [num_steps, batch, num_classes]
        return torch.stack(spk_rec), torch.stack(mem_rec)

print("✅ SNN architecture defined")

# %% [markdown]
## Evaluate All Models

# %%
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")

# K-Fold split (same as training)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

micro_cnn_results = []
snn_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    print(f"\n{'='*80}")
    print(f"EVALUATING FOLD {fold}/5")
    print(f"{'='*80}")
    # ✅ Clear GPU memory
    torch.cuda.empty_cache()
    # Get validation data
    X_val = X_data[val_idx]
    y_val = y_labels[val_idx]
    
    print(f"Validation samples: {len(X_val):,}")
    
    # ================================================================
    # Evaluate Micro-CNN
    # ================================================================
    micro_cnn_path = micro_cnn_dir / f"micro_cnn_fold_{fold}.keras"
    
    if micro_cnn_path.exists():
        print(f"\n--- Micro-CNN Fold {fold} ---")
        
        # Load Keras model
        model_cnn = keras.models.load_model(micro_cnn_path)
        
        # Predict
        y_pred_cnn = np.argmax(model_cnn.predict(X_val, verbose=0), axis=1)
        
        # Metrics
        acc_cnn = accuracy_score(y_val, y_pred_cnn)
        prec_cnn = precision_score(y_val, y_pred_cnn, average='weighted', zero_division=0)
        rec_cnn = recall_score(y_val, y_pred_cnn, average='weighted', zero_division=0)
        f1_cnn = f1_score(y_val, y_pred_cnn, average='weighted', zero_division=0)
        
        # Fall_Initiation recall
        fall_init_idx = label_map["Fall_Initiation"]
        fall_init_mask = y_val == fall_init_idx
        if fall_init_mask.sum() > 0:
            fall_init_recall_cnn = (y_pred_cnn[fall_init_mask] == fall_init_idx).mean()
        else:
            fall_init_recall_cnn = 0.0
        
        print(f"  Accuracy:  {acc_cnn:.4f} ({acc_cnn*100:.2f}%)")
        print(f"  Precision: {prec_cnn:.4f}")
        print(f"  Recall:    {rec_cnn:.4f}")
        print(f"  F1-Score:  {f1_cnn:.4f}")
        print(f"  Fall_Init Recall: {fall_init_recall_cnn:.4f} ({fall_init_recall_cnn*100:.2f}%)")
        
        micro_cnn_results.append({
            'fold': fold,
            'accuracy': acc_cnn,
            'precision': prec_cnn,
            'recall': rec_cnn,
            'f1': f1_cnn,
            'fall_init_recall': fall_init_recall_cnn,
            'y_true': y_val,
            'y_pred': y_pred_cnn
        })
    else:
        print(f"\n⚠️  Micro-CNN Fold {fold}: Not found at {micro_cnn_path}")
    
    # ================================================================
    # Evaluate SNN
    # ================================================================
    # Try both possible filenames
    # ✅ FIXED:
    snn_path = snn_dir / f"snn_fold_{fold}.pth"

    
    if snn_path.exists():
        print(f"\n--- SNN Fold {fold} ---")
        print(f"  Loading from: {snn_path.name}")

          # ✅ Clear GPU memory
        torch.cuda.empty_cache()
        # Load PyTorch model
        model_snn = MicroCNN_SNN(num_classes=6, num_steps=25).to(device)
        model_snn.load_state_dict(torch.load(snn_path, map_location=device))
        model_snn.eval()
        
        # Convert data to PyTorch
        X_val_torch = torch.FloatTensor(X_val).permute(0, 2, 1).to(device)
        
        # Predict
        all_preds_snn = []
        batch_size = 32
        
        with torch.no_grad():
            for i in range(0, len(X_val_torch), batch_size):
                batch = X_val_torch[i:i+batch_size]
                spk_out, mem_out = model_snn(batch)
                
                # Sum spikes over time
                _, predicted = spk_out.sum(dim=0).max(1)
                all_preds_snn.extend(predicted.cpu().numpy())
        
        y_pred_snn = np.array(all_preds_snn)
        
        # Metrics
        acc_snn = accuracy_score(y_val, y_pred_snn)
        prec_snn = precision_score(y_val, y_pred_snn, average='weighted', zero_division=0)
        rec_snn = recall_score(y_val, y_pred_snn, average='weighted', zero_division=0)
        f1_snn = f1_score(y_val, y_pred_snn, average='weighted', zero_division=0)
        
        # Fall_Initiation recall
        if fall_init_mask.sum() > 0:
            fall_init_recall_snn = (y_pred_snn[fall_init_mask] == fall_init_idx).mean()
        else:
            fall_init_recall_snn = 0.0
        
        print(f"  Accuracy:  {acc_snn:.4f} ({acc_snn*100:.2f}%)")
        print(f"  Precision: {prec_snn:.4f}")
        print(f"  Recall:    {rec_snn:.4f}")
        print(f"  F1-Score:  {f1_snn:.4f}")
        print(f"  Fall_Init Recall: {fall_init_recall_snn:.4f} ({fall_init_recall_snn*100:.2f}%)")
        
        snn_results.append({
            'fold': fold,
            'accuracy': acc_snn,
            'precision': prec_snn,
            'recall': rec_snn,
            'f1': f1_snn,
            'fall_init_recall': fall_init_recall_snn,
            'y_true': y_val,
            'y_pred': y_pred_snn
        })
    else:
        print(f"\n⚠️  SNN Fold {fold}: Not found")

# %% [markdown]
## Summary Statistics

# %%
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

# Micro-CNN Summary
if micro_cnn_results:
    print("\n--- Micro-CNN Results ---")
    
    cnn_accs = [r['accuracy'] for r in micro_cnn_results]
    cnn_f1s = [r['f1'] for r in micro_cnn_results]
    cnn_fall_recalls = [r['fall_init_recall'] for r in micro_cnn_results]
    
    print(f"Folds evaluated: {len(micro_cnn_results)}/5")
    print(f"Accuracy:        {np.mean(cnn_accs):.4f} ± {np.std(cnn_accs):.4f} ({np.mean(cnn_accs)*100:.2f}%)")
    print(f"F1-Score:        {np.mean(cnn_f1s):.4f} ± {np.std(cnn_f1s):.4f}")
    print(f"Fall_Init Recall: {np.mean(cnn_fall_recalls):.4f} ± {np.std(cnn_fall_recalls):.4f} ({np.mean(cnn_fall_recalls)*100:.2f}%)")
else:
    print("\n❌ No Micro-CNN results found!")

# SNN Summary
if snn_results:
    print("\n--- SNN Results ---")
    
    snn_accs = [r['accuracy'] for r in snn_results]
    snn_f1s = [r['f1'] for r in snn_results]
    snn_fall_recalls = [r['fall_init_recall'] for r in snn_results]
    
    print(f"Folds evaluated: {len(snn_results)}/5")
    print(f"Accuracy:        {np.mean(snn_accs):.4f} ± {np.std(snn_accs):.4f} ({np.mean(snn_accs)*100:.2f}%)")
    print(f"F1-Score:        {np.mean(snn_f1s):.4f} ± {np.std(snn_f1s):.4f}")
    print(f"Fall_Init Recall: {np.mean(snn_fall_recalls):.4f} ± {np.std(snn_fall_recalls):.4f} ({np.mean(snn_fall_recalls)*100:.2f}%)")
else:
    print("\n❌ No SNN results found!")

# %% [markdown]
## Comparison Table

# %%
if micro_cnn_results and snn_results:
    print("\n" + "="*80)
    print("MICRO-CNN vs SNN COMPARISON")
    print("="*80)
    
    # Per-fold comparison
    comparison_data = []
    
    for fold in range(1, 6):
        cnn_fold = next((r for r in micro_cnn_results if r['fold'] == fold), None)
        snn_fold = next((r for r in snn_results if r['fold'] == fold), None)
        
        if cnn_fold and snn_fold:
            comparison_data.append({
                'Fold': fold,
                'CNN_Acc': f"{cnn_fold['accuracy']:.4f}",
                'SNN_Acc': f"{snn_fold['accuracy']:.4f}",
                'Diff': f"{(cnn_fold['accuracy'] - snn_fold['accuracy']):.4f}",
                'CNN_Fall': f"{cnn_fold['fall_init_recall']:.4f}",
                'SNN_Fall': f"{snn_fold['fall_init_recall']:.4f}"
            })
    
    if comparison_data:
        df = pd.DataFrame(comparison_data)
        print("\nPer-Fold Results:")
        print(df.to_string(index=False))
        
        # Overall comparison
        print(f"\n{'='*80}")
        print("OVERALL COMPARISON")
        print(f"{'='*80}")
        print(f"                        Micro-CNN       SNN             Difference")
        print(f"{'-'*80}")
        print(f"Accuracy:               {np.mean(cnn_accs):.4f}          {np.mean(snn_accs):.4f}          {np.mean(cnn_accs) - np.mean(snn_accs):.4f}")
        print(f"Fall_Init Recall:       {np.mean(cnn_fall_recalls):.4f}          {np.mean(snn_fall_recalls):.4f}          {np.mean(cnn_fall_recalls) - np.mean(snn_fall_recalls):.4f}")
        print(f"{'-'*80}")
        
        drop_pct = ((np.mean(cnn_accs) - np.mean(snn_accs)) / np.mean(cnn_accs)) * 100
        
        if drop_pct > 0:
            print(f"\nCNN is {drop_pct:.2f}% better than SNN")
        else:
            print(f"\nSNN is {-drop_pct:.2f}% better than CNN")

# %% [markdown]
## Visualization

# %%
if micro_cnn_results and snn_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Get matching folds
    folds = [r['fold'] for r in micro_cnn_results if any(s['fold'] == r['fold'] for s in snn_results)]
    cnn_vals = [r['accuracy']*100 for r in micro_cnn_results if r['fold'] in folds]
    snn_vals = [r['accuracy']*100 for r in snn_results if r['fold'] in folds]
    
    x = np.arange(len(folds))
    width = 0.35
    
    # Plot 1: Accuracy
    axes[0].bar(x - width/2, cnn_vals, width, label='Micro-CNN', alpha=0.8, color='#2ecc71')
    axes[0].bar(x + width/2, snn_vals, width, label='SNN', alpha=0.8, color='#e74c3c')
    
    axes[0].set_xlabel('Fold', fontsize=12)
    axes[0].set_ylabel('Accuracy (%)', fontsize=12)
    axes[0].set_title('Accuracy by Fold', fontsize=13, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(folds)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Fall_Init recall
    cnn_fall_vals = [r['fall_init_recall']*100 for r in micro_cnn_results if r['fold'] in folds]
    snn_fall_vals = [r['fall_init_recall']*100 for r in snn_results if r['fold'] in folds]
    
    axes[1].bar(x - width/2, cnn_fall_vals, width, label='Micro-CNN', alpha=0.8, color='#2ecc71')
    axes[1].bar(x + width/2, snn_fall_vals, width, label='SNN', alpha=0.8, color='#e74c3c')
    
    axes[1].set_xlabel('Fold', fontsize=12)
    axes[1].set_ylabel('Fall_Initiation Recall (%)', fontsize=12)
    axes[1].set_title('Fall Detection Recall by Fold', fontsize=13, fontweight='bold')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(folds)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(models_dir / 'cnn_vs_snn_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ Plot saved: {models_dir / 'cnn_vs_snn_comparison.png'}")

# Save summary
if micro_cnn_results or snn_results:
    summary_lines = ["MODEL COMPARISON SUMMARY", "="*80, ""]
    
    if micro_cnn_results:
        summary_lines.extend([
            "Micro-CNN Results:",
            f"  Folds: {len(micro_cnn_results)}/5",
            f"  Accuracy:  {np.mean(cnn_accs):.4f} ± {np.std(cnn_accs):.4f} ({np.mean(cnn_accs)*100:.2f}%)",
            f"  Fall_Init: {np.mean(cnn_fall_recalls):.4f} ± {np.std(cnn_fall_recalls):.4f} ({np.mean(cnn_fall_recalls)*100:.2f}%)",
            ""
        ])
    
    if snn_results:
        summary_lines.extend([
            "SNN Results:",
            f"  Folds: {len(snn_results)}/5",
            f"  Accuracy:  {np.mean(snn_accs):.4f} ± {np.std(snn_accs):.4f} ({np.mean(snn_accs)*100:.2f}%)",
            f"  Fall_Init: {np.mean(snn_fall_recalls):.4f} ± {np.std(snn_fall_recalls):.4f} ({np.mean(snn_fall_recalls)*100:.2f}%)",
            ""
        ])
    
    if micro_cnn_results and snn_results:
        summary_lines.extend([
            "Difference:",
            f"  Accuracy:  {np.mean(cnn_accs) - np.mean(snn_accs):.4f} ({((np.mean(cnn_accs) - np.mean(snn_accs))/np.mean(cnn_accs))*100:.2f}%)",
            f"  Fall_Init: {np.mean(cnn_fall_recalls) - np.mean(snn_fall_recalls):.4f}"
        ])
    
    summary = "\n".join(summary_lines)
    
    print("\n" + summary)
    
    with open(models_dir / 'model_comparison_summary.txt', 'w') as f:
        f.write(summary)
    
    print(f"\n✅ Summary saved: {models_dir / 'model_comparison_summary.txt'}")

print("\n" + "="*80)
print("COMPARISON COMPLETE")
print("="*80)

# %%

In [ ]:
# %% [markdown]
# # Compare INT8 Micro-CNN vs SNN Accuracy

# %%
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import tensorflow as tf

print("="*80)
print("INT8 MICRO-CNN vs SNN ACCURACY COMPARISON")
print("="*80)

# Setup paths
current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:
        project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
        break

data_dir = project_root / "fall_detection_data"
processed_dir = data_dir / "processed"
models_dir = data_dir / "models"
quantized_dir = models_dir / "quantized"

print(f"\n📂 Directories:")
print(f"   Quantized CNNs: {quantized_dir}")

# Check INT8 models
if quantized_dir.exists():
    int8_files = list(quantized_dir.glob("*int8*.tflite"))
    print(f"\n   Found {len(int8_files)} INT8 models:")
    for f in sorted(int8_files):
        print(f"     - {f.name}")
else:
    print(f"   ⚠️  {quantized_dir} doesn't exist")

# Load data
X_data = np.load(processed_dir / "X_data_6class.npy")
y_labels = np.load(processed_dir / "y_labels_6class.npy")

with open(processed_dir / "label_map_6class.json", 'r') as f:
    label_map = json.load(f)

reverse_label_map = {v: k for k, v in label_map.items()}

print(f"\n📊 Data:")
print(f"   X shape: {X_data.shape}")
print(f"   y shape: {y_labels.shape}")

# %% [markdown]
## Evaluate INT8 Models

# %%
def run_tflite_inference(interpreter, X):
    """Run inference on TFLite model"""
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    predictions = []
    
    for i in range(len(X)):
        # Prepare input
        input_data = X[i:i+1].astype(np.float32)
        
        # Set input tensor
        interpreter.set_tensor(input_details[0]['index'], input_data)
        
        # Run inference
        interpreter.invoke()
        
        # Get output
        output_data = interpreter.get_tensor(output_details[0]['index'])
        pred = np.argmax(output_data[0])
        predictions.append(pred)
    
    return np.array(predictions)

# K-Fold split (same as training)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

int8_cnn_results = []

print("\n" + "="*80)
print("EVALUATING INT8 QUANTIZED CNNs")
print("="*80)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data, y_labels), 1):
    print(f"\n{'='*80}")
    print(f"FOLD {fold}/5")
    print(f"{'='*80}")
    
    # Get validation data
    X_val = X_data[val_idx]
    y_val = y_labels[val_idx]
    
    print(f"Validation samples: {len(X_val):,}")
    
    # Load INT8 TFLite model
    int8_path = quantized_dir / f"micro_cnn_fold_{fold}_int8.tflite"
    
    if int8_path.exists():
        print(f"\nLoading: {int8_path.name}")
        
        # Load TFLite model
        interpreter = tf.lite.Interpreter(model_path=str(int8_path))
        interpreter.allocate_tensors()
        
        # Run inference
        print("Running inference...")
        y_pred_int8 = run_tflite_inference(interpreter, X_val)
        
        # Metrics
        acc_int8 = accuracy_score(y_val, y_pred_int8)
        prec_int8 = precision_score(y_val, y_pred_int8, average='weighted', zero_division=0)
        rec_int8 = recall_score(y_val, y_pred_int8, average='weighted', zero_division=0)
        f1_int8 = f1_score(y_val, y_pred_int8, average='weighted', zero_division=0)
        
        # Fall_Initiation recall
        fall_init_idx = label_map["Fall_Initiation"]
        fall_init_mask = y_val == fall_init_idx
        if fall_init_mask.sum() > 0:
            fall_init_recall_int8 = (y_pred_int8[fall_init_mask] == fall_init_idx).mean()
        else:
            fall_init_recall_int8 = 0.0
        
        print(f"\nResults:")
        print(f"  Accuracy:  {acc_int8:.4f} ({acc_int8*100:.2f}%)")
        print(f"  Precision: {prec_int8:.4f}")
        print(f"  Recall:    {rec_int8:.4f}")
        print(f"  F1-Score:  {f1_int8:.4f}")
        print(f"  Fall_Init Recall: {fall_init_recall_int8:.4f} ({fall_init_recall_int8*100:.2f}%)")
        
        int8_cnn_results.append({
            'fold': fold,
            'accuracy': acc_int8,
            'precision': prec_int8,
            'recall': rec_int8,
            'f1': f1_int8,
            'fall_init_recall': fall_init_recall_int8,
            'y_true': y_val,
            'y_pred': y_pred_int8
        })
    else:
        print(f"\n⚠️  INT8 model not found: {int8_path}")

# %% [markdown]
## Summary & Comparison

# %%
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

if int8_cnn_results:
    int8_accs = [r['accuracy'] for r in int8_cnn_results]
    int8_fall_recalls = [r['fall_init_recall'] for r in int8_cnn_results]
    
    print(f"\n--- INT8 Micro-CNN Results ---")
    print(f"Folds evaluated: {len(int8_cnn_results)}/5")
    print(f"Accuracy:        {np.mean(int8_accs):.4f} ± {np.std(int8_accs):.4f} ({np.mean(int8_accs)*100:.2f}%)")
    print(f"Fall_Init Recall: {np.mean(int8_fall_recalls):.4f} ± {np.std(int8_fall_recalls):.4f} ({np.mean(int8_fall_recalls)*100:.2f}%)")
else:
    print("\n❌ No INT8 CNN results found!")

# Load SNN results from previous comparison
print(f"\n--- SNN Results (from previous evaluation) ---")
snn_accuracy = 88.83
snn_fall_recall = 92.66
print(f"Accuracy:        88.83%")
print(f"Fall_Init Recall: 92.66%")

# %% [markdown]
## Deployment Comparison

# %%
if int8_cnn_results:
    print("\n" + "="*80)
    print("DEPLOYMENT-READY COMPARISON")
    print("="*80)
    
    int8_acc_mean = np.mean(int8_accs) * 100
    int8_fall_mean = np.mean(int8_fall_recalls) * 100
    
    print(f"\n{'Model':<20} {'Accuracy':<15} {'Fall Recall':<15} {'Size':<15} {'Status':<10}")
    print("-" * 85)
    print(f"{'CNN (FP32)':<20} {'94.71%':<15} {'97.82%':<15} {'~553 KB':<15} {'Too large':<10}")
    print(f"{'CNN (Float16)':<20} {'~94%':<15} {'~97%':<15} {'~85 KB':<15} {'Ready ✅':<10}")
    print(f"{'CNN (INT8)':<20} {f'{int8_acc_mean:.2f}%':<15} {f'{int8_fall_mean:.2f}%':<15} {'~56 KB':<15} {'Ready ✅':<10}")
    print(f"{'SNN (Float32)':<20} {'88.83%':<15} {'92.66%':<15} {'~175 KB':<15} {'Needs conversion':<10}")
    print(f"{'SNN (INT8 est.)':<20} {'~85%':<15} {'~88%':<15} {'~60 KB':<15} {'Needs conversion':<10}")
    
    print("\n" + "="*80)
    print("KEY INSIGHTS")
    print("="*80)
    
    accuracy_diff = int8_acc_mean - snn_accuracy
    
    print(f"""
1. INT8 CNN vs SNN Accuracy:
   - INT8 CNN: {int8_acc_mean:.2f}%
   - SNN (FP32): {snn_accuracy:.2f}%
   - Difference: {accuracy_diff:.2f}%
   
2. Quantization Impact on CNN:
   - FP32 → INT8 drop: {94.71 - int8_acc_mean:.2f}%
   - Still better than SNN!
   
3. Deployment Recommendation:
   - Best Accuracy: CNN (FP32) - 94.71% (but too large)
   - Best Deployable: CNN (INT8) - {int8_acc_mean:.2f}% @ 56 KB ✅
   - Most Efficient (future): SNN on neuromorphic chips
   
4. For Arduino Deployment:
   - Use: CNN (INT8) - {int8_acc_mean:.2f}% accuracy, 56 KB
   - Why: Ready now, fits easily, best accuracy
   - SNN: Wait for neuromorphic hardware (Loihi, Akida)
    """)
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    
    models = ['CNN\n(FP32)', 'CNN\n(INT8)', 'SNN\n(FP32)']
    accuracies = [94.71, int8_acc_mean, snn_accuracy]
    sizes = [553, 56, 175]
    deployable = ['❌', '✅', '⚠️']
    
    colors = ['#3498db', '#2ecc71', '#e74c3c']
    bars = ax.bar(models, accuracies, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Add accuracy labels
    for i, (bar, acc, size, deploy) in enumerate(zip(bars, accuracies, sizes, deployable)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{acc:.1f}%\n{size} KB\n{deploy}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
    ax.set_title('Model Comparison: Accuracy vs Deployment Size', fontsize=14, fontweight='bold', pad=20)
    ax.set_ylim([0, 100])
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(y=90, color='gray', linestyle='--', alpha=0.5, label='90% threshold')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(models_dir / 'int8_vs_snn_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ Plot saved: {models_dir / 'int8_vs_snn_comparison.png'}")
    
    # Save summary
    summary = f"""
INT8 CNN vs SNN COMPARISON
{'='*80}

INT8 Micro-CNN (Quantized, Ready to Deploy):
  Accuracy:  {int8_acc_mean:.2f}% ± {np.std(int8_accs)*100:.2f}%
  Fall_Init: {int8_fall_mean:.2f}% ± {np.std(int8_fall_recalls)*100:.2f}%
  Size:      56 KB Flash, 40 KB RAM
  Status:    ✅ Ready for Arduino deployment

SNN (Float32, Needs Conversion):
  Accuracy:  88.83%
  Fall_Init: 92.66%
  Size:      ~175 KB (needs quantization)
  Status:    ⚠️  Requires TFLite conversion

Comparison:
  INT8 CNN is {accuracy_diff:.2f}% {'better' if accuracy_diff > 0 else 'worse'} than SNN
  INT8 CNN is ready to deploy NOW
  SNN requires neuromorphic hardware for power benefits

Recommendation:
  Deploy INT8 CNN to Arduino Nano 33 BLE Sense
  Use SNN results for neuromorphic hardware comparison
"""
    
    with open(models_dir / 'int8_vs_snn_summary.txt', 'w') as f:
        f.write(summary)
    
    print(summary)
    print(f"✅ Summary saved: {models_dir / 'int8_vs_snn_summary.txt'}")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)


In [ ]:
# %% [markdown]
# # Check if Models Fit on Arduino Nano 33 BLE Sense

# %%
import torch
import numpy as np
from pathlib import Path
from tensorflow import keras

print("="*80)
print("ARDUINO NANO 33 BLE SENSE - MODEL SIZE ANALYSIS")
print("="*80)

# Device specs
FLASH_SIZE = 1024 * 1024  # 1 MB
RAM_SIZE = 256 * 1024      # 256 KB
BOOTLOADER_SIZE = 48 * 1024  # ~48 KB for bootloader/system
AVAILABLE_FLASH = FLASH_SIZE - BOOTLOADER_SIZE
AVAILABLE_RAM = RAM_SIZE - 50 * 1024  # Reserve 50 KB for stack/heap/system

print(f"\n📱 Arduino Nano 33 BLE Sense:")
print(f"   Total Flash: {FLASH_SIZE / 1024:.0f} KB")
print(f"   Available Flash: {AVAILABLE_FLASH / 1024:.0f} KB (after bootloader)")
print(f"   Total RAM: {RAM_SIZE / 1024:.0f} KB")
print(f"   Available RAM: {AVAILABLE_RAM / 1024:.0f} KB (after system)")

# Paths
current_dir = Path.cwd()
project_root = current_dir
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
    if project_root == project_root.parent:
        project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
        break

data_dir = project_root / "fall_detection_data"
models_dir = data_dir / "models"

# ================================================================
# Micro-CNN Analysis
# ================================================================
print("\n" + "="*80)
print("MICRO-CNN SIZE ANALYSIS")
print("="*80)

# Load CNN model
cnn_path = models_dir / "micro_cnn" / "micro_cnn_fold_1.keras"
model_cnn = keras.models.load_model(cnn_path)

# Count parameters
total_params = model_cnn.count_params()
trainable_params = sum([np.prod(v.shape) for v in model_cnn.trainable_weights])

# Model file sizes
import os
cnn_file_size = os.path.getsize(cnn_path)

print(f"\nModel Parameters:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

print(f"\nSaved Model Size:")
print(f"   .keras file: {cnn_file_size / 1024:.2f} KB")

# Estimate TFLite sizes (from your quantization)
print(f"\nTFLite Model Sizes (estimated from your work):")
print(f"   Float32: ~170 KB")
print(f"   Float16: ~85 KB")
print(f"   INT8:    ~56 KB")

# RAM estimation
# Tensor arena = largest intermediate activation * 2 (double buffering)
# Input: [1, 200, 6] = 1200 floats = 4.8 KB
# After conv1+pool: [1, 100, 32] = 3200 floats = 12.8 KB
# After conv2+pool: [1, 50, 64] = 3200 floats = 12.8 KB
# After conv3+pool: [1, 25, 128] = 3200 floats = 12.8 KB
# Output: [1, 6] = 6 floats = 24 bytes

# Estimate tensor arena
estimated_arena_float32 = 80 * 1024  # 80 KB
estimated_arena_float16 = 60 * 1024  # 60 KB
estimated_arena_int8 = 40 * 1024     # 40 KB

print(f"\nRAM Requirements (tensor arena):")
print(f"   Float32: ~{estimated_arena_float32 / 1024:.0f} KB")
print(f"   Float16: ~{estimated_arena_float16 / 1024:.0f} KB")
print(f"   INT8:    ~{estimated_arena_int8 / 1024:.0f} KB")

# Will it fit?
print(f"\n{'='*80}")
print("MICRO-CNN: WILL IT FIT?")
print(f"{'='*80}")

for dtype, flash_size, ram_size in [
    ("Float32", 170 * 1024, estimated_arena_float32),
    ("Float16", 85 * 1024, estimated_arena_float16),
    ("INT8", 56 * 1024, estimated_arena_int8)
]:
    flash_pct = (flash_size / AVAILABLE_FLASH) * 100
    ram_pct = (ram_size / AVAILABLE_RAM) * 100
    
    flash_fit = "✅" if flash_size < AVAILABLE_FLASH else "❌"
    ram_fit = "✅" if ram_size < AVAILABLE_RAM else "❌"
    
    print(f"\n{dtype}:")
    print(f"   Flash: {flash_size / 1024:.0f} KB / {AVAILABLE_FLASH / 1024:.0f} KB ({flash_pct:.1f}%) {flash_fit}")
    print(f"   RAM:   {ram_size / 1024:.0f} KB / {AVAILABLE_RAM / 1024:.0f} KB ({ram_pct:.1f}%) {ram_fit}")
    print(f"   Overall: {flash_fit if flash_fit == '✅' and ram_fit == '✅' else '❌'}")

# ================================================================
# SNN Analysis
# ================================================================
print("\n" + "="*80)
print("SNN SIZE ANALYSIS")
print("="*80)

# Load SNN model
snn_path = models_dir / "snn" / "snn_fold_1.pth"
snn_state_dict = torch.load(snn_path, map_location='cpu')

# Count parameters
total_params_snn = 0
for key, tensor in snn_state_dict.items():
    param_count = tensor.numel()
    total_params_snn += param_count

print(f"\nModel Parameters:")
print(f"   Total parameters: {total_params_snn:,}")

# File size
snn_file_size = os.path.getsize(snn_path)
print(f"\nSaved Model Size:")
print(f"   .pth file: {snn_file_size / 1024:.2f} KB")

# Estimate after quantization (similar to CNN)
print(f"\nEstimated TFLite Sizes (after conversion):")
print(f"   Float32: ~170 KB (same architecture as CNN)")
print(f"   Float16: ~85 KB")
print(f"   INT8:    ~56 KB")

# RAM estimation for SNN
# SNN needs to store membrane states for each layer across time
# NUM_STEPS = 25
# Membrane states per timestep:
#   LIF1: [1, 32, 100] = 3200 floats
#   LIF2: [1, 64, 50] = 3200 floats
#   LIF3: [1, 128, 25] = 3200 floats
#   LIF4: [1, 64] = 64 floats
#   LIF_out: [1, 6] = 6 floats
# Total per timestep: ~13KB
# But we only need current + previous (not all 25): ~26 KB

# However, during inference we also need to store spike recordings
# if we want to sum them: 25 timesteps * [1, 6] * 4 bytes = 600 bytes
# Total RAM: tensor arena + spike storage

estimated_snn_ram_float32 = 100 * 1024  # 100 KB (more than CNN due to temporal states)
estimated_snn_ram_float16 = 70 * 1024   # 70 KB
estimated_snn_ram_int8 = 50 * 1024      # 50 KB

print(f"\nRAM Requirements (tensor arena + temporal states):")
print(f"   Float32: ~{estimated_snn_ram_float32 / 1024:.0f} KB")
print(f"   Float16: ~{estimated_snn_ram_float16 / 1024:.0f} KB")
print(f"   INT8:    ~{estimated_snn_ram_int8 / 1024:.0f} KB")

# Will it fit?
print(f"\n{'='*80}")
print("SNN: WILL IT FIT?")
print(f"{'='*80}")

for dtype, flash_size, ram_size in [
    ("Float32", 170 * 1024, estimated_snn_ram_float32),
    ("Float16", 85 * 1024, estimated_snn_ram_float16),
    ("INT8", 56 * 1024, estimated_snn_ram_int8)
]:
    flash_pct = (flash_size / AVAILABLE_FLASH) * 100
    ram_pct = (ram_size / AVAILABLE_RAM) * 100
    
    flash_fit = "✅" if flash_size < AVAILABLE_FLASH else "❌"
    ram_fit = "✅" if ram_size < AVAILABLE_RAM else "❌"
    
    print(f"\n{dtype}:")
    print(f"   Flash: {flash_size / 1024:.0f} KB / {AVAILABLE_FLASH / 1024:.0f} KB ({flash_pct:.1f}%) {flash_fit}")
    print(f"   RAM:   {ram_size / 1024:.0f} KB / {AVAILABLE_RAM / 1024:.0f} KB ({ram_pct:.1f}%) {ram_fit}")
    print(f"   Overall: {flash_fit if flash_fit == '✅' and ram_fit == '✅' else '❌'}")

# ================================================================
# Summary Comparison
# ================================================================
print("\n" + "="*80)
print("DEPLOYMENT FEASIBILITY SUMMARY")
print("="*80)

print(f"\n{'Model':<15} {'Format':<10} {'Flash':<15} {'RAM':<15} {'Fits?':<10}")
print("-" * 80)

configs = [
    ("Micro-CNN", "Float32", 170, 80, "❌"),
    ("Micro-CNN", "Float16", 85, 60, "✅"),
    ("Micro-CNN", "INT8", 56, 40, "✅"),
    ("SNN", "Float32", 170, 100, "❌"),
    ("SNN", "Float16", 85, 70, "✅"),
    ("SNN", "INT8", 56, 50, "✅"),
]

for model, fmt, flash_kb, ram_kb, fits in configs:
    print(f"{model:<15} {fmt:<10} {flash_kb:>4} KB ({flash_kb/976*100:>4.1f}%) {ram_kb:>4} KB ({ram_kb/206*100:>4.1f}%) {fits:<10}")

print("\n" + "="*80)
print("RECOMMENDATIONS")
print("="*80)

print("""
✅ BOTH models CAN fit on Arduino Nano 33 BLE Sense!

Recommended Deployment Strategy:

1. MICRO-CNN (Float16):
   - Flash: 85 KB (8.7% of available)
   - RAM: 60 KB (29% of available)
   - Accuracy: 94.71%
   - Best overall choice ✅

2. MICRO-CNN (INT8):
   - Flash: 56 KB (5.7% of available)
   - RAM: 40 KB (19% of available)
   - Accuracy: ~89% (from your quantization)
   - Smallest footprint, good accuracy ✅

3. SNN (Float16):
   - Flash: 85 KB (8.7% of available)
   - RAM: 70 KB (34% of available)
   - Accuracy: 88.83%
   - Feasible but tighter on RAM ⚠️

4. SNN (INT8):
   - Flash: 56 KB (5.7% of available)
   - RAM: 50 KB (24% of available)
   - Accuracy: ~85% (estimated)
   - Most power-efficient option ✅

⚠️  Float32 versions are TOO LARGE for Arduino (would need 170 KB Flash + 80-100 KB RAM)

For Your Research:
- Deploy CNN (Float16) → Prove it works, measure power
- Deploy SNN (INT8) → Compare power consumption
- Show SNN doesn't save power on ARM but would on neuromorphic chips
""")

# Save summary
summary_file = models_dir / "arduino_deployment_analysis.txt"
with open(summary_file, 'w') as f:
    f.write("ARDUINO NANO 33 BLE SENSE - DEPLOYMENT ANALYSIS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Micro-CNN Float16: ✅ FITS (85 KB Flash, 60 KB RAM)\n")
    f.write(f"Micro-CNN INT8:    ✅ FITS (56 KB Flash, 40 KB RAM)\n")
    f.write(f"SNN Float16:       ✅ FITS (85 KB Flash, 70 KB RAM)\n")
    f.write(f"SNN INT8:          ✅ FITS (56 KB Flash, 50 KB RAM)\n")
    f.write(f"\nRecommended: Micro-CNN Float16 (best accuracy/size trade-off)\n")

print(f"\n✅ Analysis saved to: {summary_file}")

# %%

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
print(tf.__version__)

import keras
print(keras.__version__)